In [1]:
import os
os.getcwd()
import pandas as pd

# Load full corpus
df = pd.read_csv("/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/nvidia_layout_all/corpus_pages_cleaned.csv")

df_amz = df.copy()

In [2]:
import re
import pandas as pd

def clean_page_text(text, patterns):
    """
    Remove all regex patterns from a given text.
    Patterns must be a list of regex strings.
    """
    if pd.isna(text):
        return text
    
    cleaned = text
    for pat in patterns:
        cleaned = re.sub(pat, " ", cleaned, flags=re.IGNORECASE | re.DOTALL)
    # collapse extra whitespace
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned



In [3]:
import re
import pandas as pd

def final_super_clean(text):
    if pd.isna(text):
        return text

    # 1. Remove the classic PDF replacement character
    text = text.replace("�", " ")

    # 2. Remove the explicit weird control chars seen in reports
    text = text.replace("\x08", " ")   # BACKSPACE
    text = text.replace("\x05", " ")   # ENQUIRY

    # 3. Remove invisible Unicode characters (safe)
    text = re.sub(r"[\u200B-\u200D\uFEFF\u00AD]", " ", text)

    # 4. Remove ALL ASCII control characters (0x00–0x1F, 0x7F)
    #    This catches any remaining PDF junk
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)

    # 5. DO NOT remove non-ASCII (we keep em-dash, accents etc.)

    # 6. Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [4]:
df["page_text_final"] = df["page_text_final"].astype(str).apply(final_super_clean)

In [5]:
patterns = [

    # ------------------------------------------------------------
    # 1. Table of contents / "Contents" ribbons
    #    2014 / 2015 / 2016 / 2017 / 2018
    # ------------------------------------------------------------
    # "Contents Select Performance Highlights Our Company Our People Human Rights ..."
    r"^Contents\s+Select\s+Performance\s+Highlights\s+Our\s+(Company|People)\b.*",
    # "Contents Our approach Our approach to reporting 04 Governance of CSR ..."
    r"^Contents\s+Our\s+approach\b.*",

    # ------------------------------------------------------------
    # 2. Performance Highlights navigation
    #    e.g. "Performance Highlights Our Company Our People Human Rights Responsible sourcing Environmental Sustainability ..."
    # ------------------------------------------------------------
    r"^Performance\s+Highlights\s+Our\s+Company\s+Our\s+People\s+Human\s+Rights(?:\s+\w+)*",

    # ------------------------------------------------------------
    # 3. Responsible sourcing navigation
    #    e.g. "Responsible sourcing Our approach Principles People Planet Resources ..."
    # ------------------------------------------------------------
    r"^Responsible\s+sourcing\s+Our\s+approach\s+Principles\s+People\s+Planet\s+Resources(?:\s+\w+)*",

    # ------------------------------------------------------------
    # 4. Resources / Planet nav blocks
    #    e.g. "Resources Planet Our approach Principles People Climate and energy ..."
    #         "Resources Planet Our approach Principles People Our environmental ..."
    # ------------------------------------------------------------
    r"^Resources\s+Planet\s+Our\s+approach\s+Principles\s+People\s+(?:Climate\s+and\s+energy|Our\s+environmental(?:\s+\w+)*).*",

    # ------------------------------------------------------------
    # 5. “Our approach…” nav ribbons
    #    e.g. "Our approach Empowering people Strengthening communities Protecting our planet ..."
    # ------------------------------------------------------------
    r"^Our\s+approach\s+Empowering\s+people\s+Strengthening\s+communities\s+Protecting\s+our\s+planet(?:\s+\w+)*",

    # Sometimes appears as "Our approach Planet Resources Principles People ..."
    r"^Our\s+approach\s+Planet\s+Resources\s+Principles\s+People(?:\s+\w+)*",

    # ------------------------------------------------------------
    # 6. 2024 Microsoft sustainability ribbons
    #    e.g. "Overview 05 Microsoft sustainability Customer sustainability Global sustainability Appendix ..."
    #         "Microsoft sustainability Customer sustainability Global sustainability Appendix ..."
    # ------------------------------------------------------------
    r"^Overview\s+\d+\s+Microsoft\s+sustainability\s+Customer\s+sustainability\s+Global\s+sustainability\s+Appendix\b.*",
    r"^Microsoft\s+sustainability\s+Customer\s+sustainability\s+Global\s+sustainability\s+Appendix\b.*",

    # Also captures broken layout version: "Overvie w O Customer sustainability Microsoft sustainability ..."
    r"^Overvie\w*\s+.*Customer\s+sustainability\s+Microsoft\s+sustainability.*",

    # ------------------------------------------------------------
    # 7. “Get the bigger picture” nav block (meta-navigation)
    #    e.g. "Get the bigger picture Explore key resources to learn more about ..."
    # ------------------------------------------------------------
    r"^Get\s+the\s+bigger\s+picture\s+Explore\s+key\s+resources\s+to\s+learn\s+more\s+about.*",

    # ------------------------------------------------------------
    # 8. Legal disclaimer footer
    #    e.g. "This document is provided as is. Information and views expressed ..."
    # ------------------------------------------------------------
    r"^This\s+document\s+is\s+provided\s+as\s+is\.\s+Information\s+and\s+views\s+expressed.*",

    # ------------------------------------------------------------
    # 9. Report footers (CSR / Citizenship / Sustainability)
    #    e.g. "Microsoft 2015 Citizenship Report"
    #         "Microsoft 2018 Corporate Social Responsibility Report 60"
    # ------------------------------------------------------------
    r"Microsoft\s+20\d{2}\s+(Citizenship|Corporate\s+Social\s+Responsibility|Sustainability)\s+Report(?:\s+\d+)?",

    # Sometimes appears at very end of the string
    r"Microsoft\s+20\d{2}\s+(Citizenship|Corporate\s+Social\s+Responsibility|Sustainability)\s+Report\s*$",

    # ------------------------------------------------------------
    # 10. Leading page numbers / section numbers (page artifacts)
    #     e.g. "02 In this report  Carbon Waste Our approach ..."
    #          " 1.5 About This Report 1 Our Company 1.1 Our Business ..."
    #          "10  10  Getting to carbon negative Amid this opportunity ..."
    # ------------------------------------------------------------
    # Strip a leading integer or integer.section with following space, at very start:
    r"^\s*\d+(?:\.\d+)?\s+",

    # Extra-safe: handle duplicated page numbers at start (e.g. "10  10  Getting ...")
    r"^\s*\d+\s+\d+\s+(?=[A-Za-z])",
]


In [6]:
df_amz["page_text_final_clean"] = df_amz["page_text_final"].apply(
    lambda x: clean_page_text(x, patterns)
)


In [7]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 600)
pd.set_option('display.max_colwidth', None)
df_amz[["page", "page_text_final", "page_text_final_clean"]]

page  \
0       0   
1       1   
2       2   
3       3   
4       4   
..    ...   
668    35   
669    36   
670    37   
671    38   
672    39   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [8]:
df_amz.shape
df_amz = df_amz.dropna(
    subset=["page_text_final", "page_text_final_clean"]
).copy()
df_amz = df_amz[df_amz["page_text_final_clean"].notna()].copy()

df_amz.shape
df_amz["page_text_final"] = df_amz["page_text_final_clean"]
df_amz = df_amz.drop(columns=["page_text_final_clean"])
df_amz = df_amz.rename(columns={"page_text_final": "page_text"})
df_amz.to_csv(
    "/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/nvidia_layout_all/corpus_pages_clean_final.csv",
    index=False
)